In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms as T
from data_loader import CelebA
from utils.load_train_setting import *
from network.Network import *
# from network.Newwork import *
from network import Encoder_MP
from network import Decoder
import os
from datetime import datetime
from utils import *
from tqdm import tqdm
import torch
import random
import numpy as np
import time
# from network.autoencoder import VQModel  # 或 AutoencoderKL
from network.autoencoder import PlainAE
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
# from skimage.metrics import structural_similarity as compare_ssim
from kornia.losses import SSIMLoss
from network.vae import VQModel,AutoencoderKL

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:

train_image_dir = "/home/ldy/..workspace/zhou/repair/latent-diffusion/data/CelebA-train/images"
val_image_dir = "/home/ldy/..workspace/zhou/repair/latent-diffusion/data/CelebA-val/images"
train_attr_path = "/home/ldy/..workspace/zhou/repair/latent-diffusion/data/CelebA-train/list_attr_celeba5001-15000.txt"
val_attr_path = "/home/ldy/..workspace/zhou/repair/latent-diffusion/data/CelebA-val/list_attr_celeba5000.txt"
# 将数据集规模减半
train_subset_size = len(train_dataset)//2
train_indices = list(range(train_subset_size))
train_subset = torch.utils.data.Subset(train_dataset, train_indices)
train_dataloader = DataLoader(train_subset, batch_size=4, shuffle=True, num_workers=4)

In [ ]:
stargan_layer = StarGAN(g_conv_dim=64, c_dim=5, g_repeat_num=6, device=device,
            model_path="/home/ldy/..workspace/zhou/repair/models/200000-G.ckpt")

In [ ]:
class EncoderDecoder(nn.Module):
	'''
	A Sequential of Encoder_MP-Noise-Decoder
	'''

	def __init__(self, H, W, message_length, noise_layers):
		super(EncoderDecoder, self).__init__()
		self.encoder = Encoder_MP(H, W, message_length)
		self.noise = Noise(noise_layers)
		self.decoder = Decoder(H, W, message_length)

	def encode_to_image(self, image, message):
		encoded_image = self.encoder(image, message)
		return encoded_image
	
	def decode_from_image(self, encoded_image):
		message = self.decoder(encoded_image)
		return message

	def forward(self, image, message):
		encoded_image = self.encoder(image, message)
		noised_image = self.noise([encoded_image, image])
		decoded_message = self.decoder(noised_image)
		return encoded_image, noised_image, decoded_message

In [ ]:
EncoderDecoder = EncoderDecoder
H, W = 128, 128

In [ ]:
for _, (images, labels) in tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc="data preparing"):
    images = images.to(device)
    clean_message = torch.Tensor(np.random.choice([0,1], (images.shape[0], message_length))).to(device)
    with torch.no_grad():
        encoded_image = encodeto_image(images, clean_message)  # 使用 .module 访问方法
    stargan_results = stargan_layer((encoded_image, images, labels))